# 🔄 Agent-to-Agent: Hierarchical Research with Bigdata.com

This notebook demonstrates an **Agent-to-Agent** architecture where:

1. **Primary Agent** first checks internal sources (database, research documents)
2. **Escalates** to Bigdata.com Research Agent for complex questions requiring external data

## Architecture

![Agent to Bigdata Research Agent](./static/agent_to_research_small.jpg)

## Use Cases Covered

| Role | Example Questions |
|------|------------------|
| **Equity Research** | Investment thesis validation, competitive analysis |
| **Credit Research** | Debt covenant analysis, refinancing risks |
| **Credit Risk** | Counterparty exposure, default probability drivers |

---

## Langsmith Tracing

![LangSmith Tracing](./static/langsmith.png)

## 1️⃣ Install Dependencies

In [1]:
%pip install langchain langchain-openai langchain-community faiss-cpu requests python-dotenv -q


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2️⃣ Setup Environment

In [2]:
from langgraph_core import setup_environment

# Setup with LangSmith tracing
config = setup_environment(
    langsmith_project='bigdata-agent-to-agent',
    enable_tracing=True
)

print("\n🔗 View agent traces: https://smith.langchain.com")

/Users/bakulkumarkakadiya/dev/github/bigdata-cookbook/Index_MA_Activity_Report/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ LangSmith tracing enabled → Project: bigdata-agent-to-agent
✅ Bigdata API Key: bd_v2_ONWV...
✅ OpenAI API Key: sk-proj--w...

🔗 View agent traces: https://smith.langchain.com


## 3️⃣ Initialize Data Sources

Create the internal database and vector store with sample financial data.

In [3]:
from langgraph_core import create_financial_database, create_vector_store

# Create SQLite database with portfolios, holdings, transactions
create_financial_database()

# Create vector store with internal research documents
create_vector_store()

print("\n📊 Sample portfolios created:")
print("   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)")
print("   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)")
print("   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)")

✅ Created 3 accounts
✅ Created 3 portfolios
✅ Created 15 holdings
✅ Created 100 transactions
✅ Created vector store with 6 documents

📊 Sample portfolios created:
   • PF001: US Large Cap Growth (AAPL, META, AMZN, GOOGL, MSFT)
   • PF002: AI & Semiconductor Focus (NVDA, AMD, AVGO, MRVL, TSM)
   • PF003: Diversified Tech Leaders (NVDA, AAPL, MSFT, CRM, ORCL)


## 4️⃣ Create Hierarchical Agent

This agent is configured to:
1. Check **internal sources first** (faster, proprietary data)
2. Use **quick external lookups** for company info
3. **Escalate** to Research Agent only when needed

In [4]:
from langgraph_core import create_hierarchical_agent

# Create agent with hierarchical tool priority
agent = create_hierarchical_agent(include_research_agent=True)

✅ Hierarchical agent created with 5 tools:
   Internal: ['internal_query_database', 'internal_portfolio_summary', 'internal_search_research']
   External: ['bigdata_lookup_company', 'bigdata_research_agent']


---

# 📈 Equity Research Use Cases

Questions that equity analysts typically ask.

### Query 1: Internal Holdings Check (No Escalation Expected)

Simple portfolio question - should use only internal tools.

In [5]:
from langgraph_core import display_agent_response

display_agent_response(agent, """
What is our total exposure to NVIDIA across all portfolios? 
Include position sizes, average cost basis, and unrealized P&L.
""", show_json=True)

🔧 internal_query_database: {
  "sql_query": "SELECT portfolio_id, shares, avg_cost, market_value, unrealized_pnl FROM holdings WHERE ticker = 'NVDA'"
}...


Our total exposure to NVIDIA across all portfolios is as follows:

1. **Portfolio PF002 (AI & Semiconductor Focus):**
   - Shares: 12,000
   - Average Cost Basis: $450.00
   - Market Value: $10,506,000
   - Unrealized P&L: $5,106,000

2. **Portfolio PF003 (Diversified Tech Leaders):**
   - Shares: 8,000
   - Average Cost Basis: $520.00
   - Market Value: $7,004,000
   - Unrealized P&L: $2,844,000

**Total Exposure:**
- Total Shares: 20,000
- Combined Market Value: $17,510,000
- Total Unrealized P&L: $7,950,000

This data is sourced from our internal database.

{'response': 'Our total exposure to NVIDIA across all portfolios is as follows:\n\n1. **Portfolio PF002 (AI & Semiconductor Focus):**\n   - Shares: 12,000\n   - Average Cost Basis: $450.00\n   - Market Value: $10,506,000\n   - Unrealized P&L: $5,106,000\n\n2. **Portfolio PF003 (Diversified Tech Leaders):**\n   - Shares: 8,000\n   - Average Cost Basis: $520.00\n   - Market Value: $7,004,000\n   - Unrealized P&L: $2,844,000\n\n**Total Exposure:**\n- Total Shares: 20,000\n- Combined Market Value: $17,510,000\n- Total Unrealized P&L: $7,950,000\n\nThis data is sourced from our internal database.',
 'tools_count': 1,
 'tools': [{'name': 'internal_query_database',
   'args': {'sql_query': "SELECT portfolio_id, shares, avg_cost, market_value, unrealized_pnl FROM holdings WHERE ticker = 'NVDA'"}}],
 'tool_results': ['{\n  "query": "SELECT portfolio_id, shares, avg_cost, market_value, unrealized_pnl FROM holdings WHERE ticker = \'NVDA\'",\n  "row_count": 2,\n  "results": [\n    {\n      "portfo

### Query 2: Investment Thesis Validation (Internal + External)

Combines internal research with market data - may escalate for current news.

In [6]:
display_agent_response(agent, """
Review our investment thesis for NVIDIA:
1. What does our internal research say about NVIDIA's competitive position?
2. What recent market developments might affect this thesis?
3. Should we adjust our position based on current information?
""")

🔧 internal_search_research: {
  "query": "NVIDIA competitive position",
  "top_k": 3
}...
🔧 internal_query_database: {
  "sql_query": "SELECT * FROM holdings WHERE company_name = 'NVIDIA'"
}...
🔧 bigdata_research_agent: {
  "query": "Recent market developments affecting NVIDIA",
  "research_effort": "standard"
}...


### Internal Research on NVIDIA's Competitive Position

According to our internal research, NVIDIA is a leading player in the semiconductor space, particularly in AI and data center markets. Key points from our investment thesis include:

1. **Data Center Revenue**: NVIDIA's data center revenue has seen a significant increase, driven by high demand for its H100/H200 GPUs used in AI training.
2. **Next-Gen Architecture**: The upcoming Blackwell architecture (B100/B200 GPUs) is expected to launch in Q2 2025, offering 2.5x performance improvements.
3. **Software Ecosystem**: NVIDIA's CUDA ecosystem, with over 4 million developers, creates substantial switching costs, reinforcing its competitive moat.
4. **AI Inference Market**: The total addressable market (TAM) for AI inference is projected to reach $150 billion by 2027, as enterprises scale AI deployments.

However, there are risk factors such as China export restrictions, competition from AMD, and potential supply constraints [From internal research].

### Recent Market Developments Affecting NVIDIA

Recent market developments impacting NVIDIA include:

1. **AI Dominance**: NVIDIA continues to lead in AI infrastructure, with new architectures like Rubin (R100) extending its market position through at least 2027 [1].
2. **Geopolitical Challenges**: China has blocked imports of NVIDIA's H200 AI chips, which could pose long-term growth challenges. Revenue from China has dropped significantly, and NVIDIA's market share in China's AI processor market is expected to decrease drastically due to export restrictions and rising domestic competition [4][5][6].
3. **Increasing Competition**: Major customers like Google and Meta are developing their own AI chips, which could erode NVIDIA's market share [7][8].
4. **Stock Performance**: Despite strong business fundamentals, NVIDIA's stock has been relatively stagnant recently, with concerns about overvaluation and an "AI bubble" [9][10].

### Position Adjustment Recommendation

Given the strong competitive position and growth prospects in AI, but considering the geopolitical risks and increasing competition, it may be prudent to closely monitor NVIDIA's performance, especially in light of upcoming earnings and the rollout of new architectures. While the internal strategy memo suggests increasing NVIDIA's weight in the portfolio due to high AI training demand, the external challenges warrant a cautious approach. It might be beneficial to maintain the current position while keeping an eye on geopolitical developments and competitive dynamics [From internal research and external sources].

### Query 3: Competitive Analysis (Escalation Expected)

Deep competitive analysis requires external research agent.

In [7]:
display_agent_response(agent, """
Perform a competitive analysis of AMD vs NVIDIA in the AI accelerator market:

1. First check: Do we have any internal research comparing these companies?
2. Then analyze: Market share trends, product roadmaps, and customer wins
3. Conclude: Which company is better positioned for 2025-2026?

Cite specific sources for your analysis.
""")

🔧 internal_search_research: {
  "query": "AMD vs NVIDIA AI accelerator market",
  "top_k": 3
}...
🔧 internal_search_research: {
  "query": "AMD AI accelerator market analysis",
  "top_k": 3
}...
🔧 internal_search_research: {
  "query": "NVIDIA AI accelerator market analysis",
  "top_k": 3
}...
🔧 bigdata_research_agent: {
  "query": "Analyze the AI accelerator market focusing on AMD and NVIDIA. Include market share trends, product roadmaps, and recent customer wins. C...


### Competitive Analysis: AMD vs NVIDIA in the AI Accelerator Market

#### Internal Research Insights
1. **AMD**:
   - **MI300X GPU Performance**: Features 192GB HBM3 memory, surpassing NVIDIA's H100 in memory capacity. Strong performance in inference for large language models (LLMs) with confirmed deployments in Microsoft Azure and Oracle Cloud. AMD targets over $5 billion in AI GPU revenue by 2025 [From internal research].
   - **Challenges**: The ROCm software ecosystem lags behind NVIDIA's CUDA, and NVIDIA maintains a mindshare advantage among AI developers [From internal research].

2. **NVIDIA**:
   - **Data Center Revenue**: Achieved $18.4 billion in Q4 2024, driven by H100/H200 GPU demand for AI training. The upcoming Blackwell architecture promises 2.5x performance improvements [From internal research].
   - **Software Moat**: The CUDA ecosystem has over 4 million developers, creating significant switching costs [From internal research].

#### External Research Analysis
1. **Market Share Trends**:
   - **NVIDIA**: Dominates the AI accelerator market with an estimated 80% to 95% market share, driven by its early entry and comprehensive ecosystem [1][2][3][4][5].
   - **AMD**: Gaining momentum as hyperscale customers like Alphabet, Meta, and Amazon seek diversification. AMD aims for a 60% CAGR in its data center segment and targets 20% market share by 2027 [6][7][8][9][10][11][12][13].

2. **Product Roadmaps**:
   - **NVIDIA**: 
     - **Blackwell Architecture**: Current revenue driver with full production ramp-up.
     - **Rubin Architecture**: Launching in late 2026, promises 5x performance improvements over Blackwell [14][15][16][17][18][19][20][21].
   - **AMD**:
     - **Instinct MI300 Series**: Already shipping to customers.
     - **Instinct MI400 Series**: Includes MI450, expected to match or surpass NVIDIA's Rubin performance [22][23][24][25][26][27][28][29][30][31][32][33][34][35][36][37][38].

3. **Customer Wins and Partnerships**:
   - **NVIDIA**: Strong relationships with major hyperscalers and partnerships with OpenAI, Oracle, and Microsoft [42][43][44][45][46][47][48][49][50][51][52][53][54][55][56][57][58][59][60].
   - **AMD**: Secured multi-year deals with OpenAI and Oracle, and partnerships with Microsoft, Meta, and HPE [27][61][62][63][64][65][66][67][68][69][70][71][72][73][74][75][76][77][78][79][80][81][82][83][84][85][86][87][88][89][90][91][92][93][94][95][96][97].

#### Conclusion: Positioning for 2025-2026
While **NVIDIA** remains a dominant force, **AMD** is better positioned for significant growth and market share expansion in 2025-2026. AMD's aggressive product roadmap, strategic customer wins, and open ecosystem approach make it a formidable competitor. The industry's move towards diversification and AMD's robust pipeline and partnerships place it in an excellent position to capture substantial market share [4][5][11][28][38][98].

**Sources**: Internal research documents and external citations [1-25].

---

# 💳 Credit Research Use Cases

Questions relevant to fixed income and credit analysts.

### Query 4: Debt and Leverage Analysis (Escalation Expected)

Credit-focused questions typically need external data.

In [8]:
display_agent_response(agent, """
Analyze the credit profile of Intel Corporation:

1. Current debt levels and maturity schedule
2. Cash flow coverage ratios and liquidity position
3. Recent credit rating actions or outlook changes
4. Key risks that could affect their investment-grade status

Note: We don't hold Intel, so you may need external sources.
""")

🔧 bigdata_research_agent: {
  "query": "Analyze the credit profile of Intel Corporation, including current debt levels and maturity schedule, cash flow coverage ratios and liqu...


Here's a comprehensive analysis of Intel Corporation's credit profile:

### Current Debt Levels and Maturity Schedule
- **Total Debt:** As of September 27, 2025, Intel's total debt was $46,553 million, down from $50,011 million at the end of 2024. This includes $2,496 million in short-term debt and $44,057 million in long-term debt [1].
- **Maturity Schedule:** 
  - **2025:** $3,750 million (reduced by $1.5 billion settled in Q1 2025 and $2.3 billion settled in Q3 2025)
  - **2026:** $2,500 million
  - **2027:** $3,826 million
  - **2028:** $3,173 million
  - **2029:** $3,288 million
  - **2030 and thereafter:** $34,448 million
  - Total outstanding principal as of December 28, 2024, was $50,985 million [2].

### Cash Flow Coverage Ratios and Liquidity Position
- **Liquidity Position:** As of September 27, 2025, Intel had cash and cash equivalents of $11,141 million, up from $8,249 million at the end of 2024. Short-term investments increased to $19,794 million, bringing total cash and short-term investments to $30,935 million [1].
- **Operating Cash Flow:** For the nine months ended September 27, 2025, net cash provided by operating activities was $5,409 million, up from $5,123 million in the prior year's period. For the full year 2024, operating cash flow was $8,288 million [2].
- **Credit Facilities:** Intel has authorization to borrow up to $10.0 billion under its commercial paper program and committed revolving credit facilities totaling $12.0 billion. There were no outstanding borrowings on these facilities as of September 27, 2025 [1].
- **Free Cash Flow:** Adjusted free cash flow for 2024 was negative $(2,228) million, an improvement from $(11,853) million in 2023 [2].

### Recent Credit Rating Actions or Outlook Changes
- **Fitch Ratings:** Intel is rated 'BBB' with a Negative outlook as of January 6, 2026. The "Negative" outlook suggests potential for a downgrade due to a weaker financial profile and lower revenue growth prospects compared to peers [3].
- **Downgrade:** In August 2025, a major credit rating agency downgraded Intel's corporate credit rating from BBB+ to BBB, citing execution risks, delayed deleveraging, and weaker demand [1].

### Key Risks Affecting Investment-Grade Status
1. **High Competition and Technological Change:** Rapid advancements and evolving customer needs in the semiconductor industry could weaken Intel's position [2].
2. **Foundry Strategy Risks:** Intel's strategy to become a major third-party foundry faces competition and significant investment risks [2].
3. **Execution Risks in Technology Roadmap:** Delays in new manufacturing technologies could lead to cost disadvantages and credit rating pressure [2].
4. **Significant Capital Investments:** High fixed costs and substantial R&D investments make Intel vulnerable to demand fluctuations [2].
5. **Geopolitical Tensions and Supply Chain Disruptions:** Global tensions and trade policies could disrupt operations and supply chains [2].
6. **Debt Obligations:** High leverage could reduce financial flexibility and lead to further downgrades if performance does not improve [2].
7. **U.S. Government Equity Ownership:** CHIPS Act funding introduces risks related to government ownership and potential legal challenges [1].
8. **Intellectual Property Risks:** Ongoing IP litigation could result in substantial fines and penalties [2].
9. **Evolving AI Market:** The shift towards GPUs in data centers could impact demand for Intel's products [2].
10. **Customer Concentration:** Reliance on a few customers makes Intel vulnerable to sales reductions [2].

These factors collectively pose challenges to Intel's financial performance and could influence future credit rating assessments.

### Query 5: Sector-Wide Credit Trends (Escalation Expected)

Macro credit analysis requires deep research.

In [9]:
display_agent_response(agent, """
What are the key credit risks facing the semiconductor sector in the current environment?

Consider:
- Capital expenditure requirements and debt financing
- Cyclical demand patterns and inventory corrections
- Geopolitical risks (US-China tensions, export controls)
- Recent bond issuances or refinancing activity

Which semiconductor companies are most vulnerable from a credit perspective?
""")

🔧 bigdata_research_agent: {
  "query": "Key credit risks facing the semiconductor sector in the current environment, including capital expenditure requirements, debt financing,...


The semiconductor sector is currently facing several key credit risks due to its capital-intensive nature, cyclical demand patterns, geopolitical tensions, and recent financial activities. Here's a detailed analysis:

### Key Credit Risks:

1. **Capital Expenditure Requirements:**
   - The semiconductor industry requires high capital investments for continuous innovation and equipment upgrades. Companies like Micron are planning significant capital expenditures, with a $20 billion budget for the current fiscal year [4][5][6][7]. The return on these investments is critical, as excessive capital expenditure without securing long-term pricing protection can erode shareholder trust and attract scrutiny from credit agencies [8].

2. **Debt Financing:**
   - To fund these capital outlays, there has been a shift towards increased debt financing. Major tech companies are raising billions in new debt to support AI and cloud infrastructure expansions [10][11][12][13][14][15][16][17][18][19][20][21]. This trend poses a risk if AI growth underperforms or takes longer to monetize, potentially challenging the ability of these companies to service their debts [19][21].

3. **Cyclical Demand Patterns:**
   - The semiconductor industry is inherently cyclical, with alternating periods of high demand and downturns [2][23][24]. While current AI-driven demand is strong, the industry remains fundamentally cyclical, with competitive pressures and aggressive pricing complicating profitability [1][2][24].

4. **Inventory Corrections:**
   - Post-pandemic, OEMs accumulated substantial inventory, and the industry is working to normalize these stockpiles [27]. Inaccurate demand forecasts can lead to excessive inventory or shortages, causing production delays and cost pressures [28][29][30].

5. **Geopolitical Risks (US-China Tensions, Export Controls):**
   - Escalating US-China tensions pose a major strategic risk [32][33][34][35]. Companies with substantial revenue exposure to China face heightened vulnerability to export restrictions, supply chain disruptions, or regulatory actions [33][35].

### Recent Bond Issuances or Refinancing Activity:

- **NXP Semiconductors** issued significant unsecured notes totaling $1.5 billion across different maturities [42].
- **ON Semiconductor** reported long-term debt of $3.35 billion and anticipates non-cash impairment and accelerated depreciation charges [43][44].

### Semiconductor Companies Most Vulnerable from a Credit Perspective:

- **NXP Semiconductors:** Recent significant bond issuances and a high leverage profile increase credit risk, especially if market conditions weaken [42].
- **ON Semiconductor:** Stable but substantial long-term debt and impending impairment charges could signal financial strain [43].
- **Taiwan Semiconductor Manufacturing Company (TSMC):** Faces significant exposure to geopolitical risks, especially US-China tensions, which could impact its revenue from China [33].
- **Companies heavily reliant on consumer-oriented markets or with less diversified product portfolios:** More susceptible to industry cyclicality and inventory corrections [46].
- **Smaller companies within the supply chain:** Limited bargaining power and susceptibility to demand fluctuations can impact their financial stability [47].

Conversely, AI chip suppliers like Broadcom and Marvell Technology appear to be in a stronger position, benefiting from robust demand and positive revenue growth projections [46].

This analysis highlights the complex landscape of credit risks in the semiconductor sector, emphasizing the need for careful financial management and strategic planning to navigate these challenges.

### Query 6: Refinancing Risk Assessment

In [10]:
display_agent_response(agent, """
Which companies in our portfolios might face refinancing challenges in 2025-2026?

1. First, check what companies we hold across all portfolios
2. For each major holding, assess:
   - Debt maturity walls
   - Current interest coverage
   - Access to capital markets
3. Flag any companies with elevated refinancing risk
""")

🔧 internal_query_database: {
  "sql_query": "SELECT DISTINCT company_name, ticker FROM holdings"
}...
🔧 internal_search_research: {
  "query": "Apple Inc. debt maturity and refinancing risk 2025-2026",
  "top_k": 1
}...
🔧 internal_search_research: {
  "query": "Microsoft Corporation debt maturity and refinancing risk 2025-2026",
  "top_k": 1
}...
🔧 internal_search_research: {
  "query": "Alphabet Inc. debt maturity and refinancing risk 2025-2026",
  "top_k": 1
}...
🔧 internal_search_research: {
  "query": "Amazon.com Inc. debt maturity and refinancing risk 2025-2026",
  "top_k": 1
}...
🔧 internal_search_research: {
  "query": "Meta Platforms Inc. debt maturity and refinancing risk 2025-2026",
  "top_k": 1
}...
🔧 internal_search_research: {
  "query": "NVIDIA Corporation debt maturity and refinancing risk 2025-2026",
  "top_k": 1
}...
🔧 internal_search_research: {
  "query": "Advanced Micro Devices debt maturity and refinancing risk 2025-2026",
  "top_k": 1
}...
🔧 internal_search_resear

Based on the internal research documents, here is a summary of the refinancing risks for the companies in our portfolios:

1. **Apple Inc. (AAPL)**, **Microsoft Corporation (MSFT)**, **Alphabet Inc. (GOOGL)**, **Amazon.com Inc. (AMZN)**, **Meta Platforms Inc. (META)**, **Broadcom Inc. (AVGO)**, **Taiwan Semiconductor (TSM)**, **Palantir Technologies (PLTR)**, **Salesforce Inc. (CRM)**, and **Oracle Corporation (ORCL)**:
   - The internal research documents did not specifically highlight refinancing risks for these companies. The focus was more on valuation risks, regulatory risks, and exposure to China. There was no mention of significant debt maturity walls or refinancing challenges in the 2025-2026 period. The documents primarily discussed broader sector risks and hedging strategies [From internal research].

2. **NVIDIA Corporation (NVDA)**:
   - The research focused on NVIDIA's strong market position and growth prospects, particularly in AI and data centers. There was no specific mention of refinancing risks or debt maturity concerns in the 2025-2026 period [From internal research].

3. **Advanced Micro Devices (AMD)**:
   - Similar to NVIDIA, the focus was on AMD's competitive positioning and growth in AI and server CPUs. There was no specific mention of refinancing risks or debt maturity concerns in the 2025-2026 period [From internal research].

Overall, the internal research did not flag any of these companies as having elevated refinancing risks in the 2025-2026 period. However, the documents primarily focused on other types of risks and opportunities, so it may be beneficial to conduct a more detailed external analysis if refinancing risk is a critical concern.

---

# ⚠️ Credit Risk Analyst Use Cases

Questions focused on counterparty risk and default analysis.

### Query 7: Counterparty Exposure Analysis (Internal First)

In [11]:
display_agent_response(agent, """
Calculate our total counterparty exposure by company:

1. Sum up all positions across portfolios for each ticker
2. Calculate as percentage of total AUM
3. Identify our top 5 concentration risks
4. Flag any single-name exposures exceeding 15% of total

Present in a risk report format.
""")

🔧 internal_query_database: {
  "sql_query": "SELECT portfolio_id, ticker, company_name, shares, current_price, market_value FROM holdings"
}...
🔧 internal_query_database: {
  "sql_query": "SELECT SUM(aum) as total_aum FROM portfolios"
}...


### Risk Report: Counterparty Exposure Analysis

#### Total Counterparty Exposure by Company

1. **Aggregate Positions Across Portfolios:**

   | Ticker | Company Name            | Total Market Value (USD) |
   |--------|-------------------------|--------------------------|
   | AAPL   | Apple Inc.              | $7,410,000               |
   | MSFT   | Microsoft Corporation   | $9,556,500               |
   | GOOGL  | Alphabet Inc.           | $876,250                 |
   | AMZN   | Amazon.com Inc.         | $1,354,500               |
   | META   | Meta Platforms Inc.     | $2,632,500               |
   | NVDA   | NVIDIA Corporation      | $17,546,000              |
   | AMD    | Advanced Micro Devices  | $1,162,000               |
   | AVGO   | Broadcom Inc.           | $2,137,500               |
   | TSM    | Taiwan Semiconductor    | $557,250                 |
   | PLTR   | Palantir Technologies   | $1,631,250               |
   | CRM    | Salesforce Inc.         | $3,255,000               |
   | ORCL   | Oracle Corporation      | $2,103,000               |

2. **Percentage of Total AUM:**

   - **Total AUM:** $95,000,000

   | Ticker | Company Name            | % of Total AUM |
   |--------|-------------------------|----------------|
   | AAPL   | Apple Inc.              | 7.80%          |
   | MSFT   | Microsoft Corporation   | 10.06%         |
   | GOOGL  | Alphabet Inc.           | 0.92%          |
   | AMZN   | Amazon.com Inc.         | 1.43%          |
   | META   | Meta Platforms Inc.     | 2.77%          |
   | NVDA   | NVIDIA Corporation      | 18.47%         |
   | AMD    | Advanced Micro Devices  | 1.22%          |
   | AVGO   | Broadcom Inc.           | 2.25%          |
   | TSM    | Taiwan Semiconductor    | 0.59%          |
   | PLTR   | Palantir Technologies   | 1.72%          |
   | CRM    | Salesforce Inc.         | 3.43%          |
   | ORCL   | Oracle Corporation      | 2.21%          |

3. **Top 5 Concentration Risks:**

   - **NVIDIA Corporation (NVDA):** 18.47%
   - **Microsoft Corporation (MSFT):** 10.06%
   - **Apple Inc. (AAPL):** 7.80%
   - **Salesforce Inc. (CRM):** 3.43%
   - **Meta Platforms Inc. (META):** 2.77%

4. **Single-Name Exposures Exceeding 15%:**

   - **NVIDIA Corporation (NVDA):** 18.47% (Flagged for high concentration risk)

#### Summary

The analysis indicates that NVIDIA Corporation represents a significant concentration risk, exceeding 15% of the total AUM. This exposure should be closely monitored and potentially rebalanced to mitigate risk. Other exposures are within acceptable limits but should be reviewed periodically to ensure alignment with risk management strategies.

**Note:** All data is sourced from internal databases.

### Query 8: ESG/Regulatory Risk (Escalation Expected)

In [12]:
display_agent_response(agent, """
Assess regulatory and ESG risks for our technology holdings:

1. Which of our holdings face significant regulatory scrutiny?
2. What recent regulatory developments could impact valuations?
3. Are there any ESG controversies affecting our portfolio companies?
4. Which positions should we consider reducing due to regulatory risk?

Focus on antitrust, data privacy, and AI governance regulations.
""")

🔧 internal_portfolio_summary: {
  "portfolio_id": "PF003"
}...
🔧 internal_search_research: {
  "query": "regulatory scrutiny NVIDIA Microsoft Apple Salesforce Oracle antitrust data privacy AI governance",
  "top_k": 3
}...
🔧 internal_search_research: {
  "query": "ESG controversies NVIDIA Microsoft Apple Salesforce Oracle",
  "top_k": 3
}...
🔧 bigdata_research_agent: {
  "query": "Recent regulatory developments impacting NVIDIA, Microsoft, Apple, Salesforce, and Oracle in terms of antitrust, data privacy, and AI go...


### Regulatory and ESG Risks for Technology Holdings

#### 1. Significant Regulatory Scrutiny
- **NVIDIA**: Faces antitrust scrutiny, particularly in China, where regulators found violations of anti-monopoly laws related to past acquisitions [1][2][3]. The U.S. has cleared some of its deals, but geopolitical tensions remain high [6][20].
- **Microsoft**: Under global antitrust scrutiny, including investigations in Switzerland, Italy, and the UK, related to licensing fees and cloud practices [37][38][39][41]. The EU is also investigating its cloud computing business [41].
- **Apple**: Faces antitrust challenges globally, including a significant case in India and fines in the EU related to its App Store practices [92][103].
- **Salesforce**: Involved in antitrust discussions through its subsidiary Slack, which led to an EU investigation against Microsoft [154].
- **Oracle**: Indirectly involved in antitrust cases related to media industry consolidation [196].

#### 2. Recent Regulatory Developments Impacting Valuations
- **NVIDIA**: Ongoing restrictions on AI chip sales to China could significantly impact revenue [20][21][25].
- **Microsoft**: Antitrust investigations could lead to changes in business practices and potential fines [37][41].
- **Apple**: Antitrust fines and investigations could affect its services revenue [92][103].
- **Salesforce**: Antitrust scrutiny related to data center energy demands could impact operations [158].
- **Oracle**: Antitrust scrutiny in the cloud market could affect its competitive positioning [209].

#### 3. ESG Controversies
- **NVIDIA**: Criticized for energy demands in AI growth and involved in a copyright lawsuit [32][36].
- **Microsoft**: Faces lawsuits related to misleading AI product pricing and data privacy issues [72][79].
- **Apple**: Under scrutiny for app store practices and environmental leadership changes [145][146].
- **Salesforce**: Criticized for AI-driven layoffs and data breaches [186][168].
- **Oracle**: Faces lawsuits related to misleading statements in debt sales and cybersecurity incidents [235][215].

#### 4. Positions to Consider Reducing Due to Regulatory Risk
- **NVIDIA**: Due to significant antitrust scrutiny and geopolitical risks related to AI chip sales [1][20].
- **Microsoft**: Given the breadth of antitrust investigations and potential regulatory changes [37][41].
- **Apple**: Due to ongoing antitrust challenges and potential fines impacting its business model [92][103].

### Conclusion
The regulatory landscape for these technology holdings is complex, with significant scrutiny in antitrust, data privacy, and AI governance. ESG controversies also pose reputational risks. It may be prudent to consider reducing positions in companies facing the most significant regulatory challenges, such as NVIDIA, Microsoft, and Apple, to mitigate potential impacts on valuations.

---

# 🔬 Custom Query

Try your own hierarchical research query:

In [13]:
# Enter your own query
custom_query = """
Compare our internal research on NVIDIA with the latest market sentiment.
Is our thesis still valid?
"""

display_agent_response(agent, custom_query)

🔧 internal_search_research: {
  "query": "NVIDIA investment thesis",
  "top_k": 3
}...
🔧 bigdata_research_agent: {
  "query": "latest market sentiment on NVIDIA",
  "research_effort": "standard"
}...


### Internal Research on NVIDIA

According to our internal research, NVIDIA remains a top pick in the semiconductor space, with a strong focus on AI and data center growth. Key points from our latest investment thesis include:

1. **Data Center Revenue**: NVIDIA's data center revenue has seen a significant increase, driven by demand for its H100/H200 GPUs for AI training.
2. **Next-Gen GPUs**: The upcoming Blackwell Architecture (B100/B200 GPUs) is expected to launch in Q2 2025, offering 2.5x performance improvements.
3. **Software Ecosystem**: NVIDIA's CUDA ecosystem, with over 4 million developers, creates significant switching costs and a strong software moat.
4. **AI Inference Opportunity**: The total addressable market (TAM) for AI inference is projected to reach $150 billion by 2027.

**Risk Factors**: Potential risks include China export restrictions, competition from AMD, and supply constraints.

**Price Target**: $950 with a "STRONG BUY" rating [From internal research].

### Latest Market Sentiment on NVIDIA

The latest market sentiment on NVIDIA is predominantly positive, driven by strong demand for its AI solutions and robust financial performance. Key drivers include:

1. **AI Demand and Market Leadership**: NVIDIA is benefiting from the booming demand for AI infrastructure, particularly in data centers. It maintains a strong competitive stance and is pivotal for modern drug discovery [1, 2, 3, 4].
2. **Financial Performance**: Recent earnings and revenue forecasts have exceeded expectations, with analysts anticipating continued high earnings growth and strong margins [5, 9, 10].
3. **Analyst Ratings**: Wall Street maintains a "Strong Buy" consensus, with many analysts increasing price targets [3, 4, 5].
4. **Product Innovation**: New platforms like Rubin and successful navigation of packaging bottlenecks contribute to optimism [3, 16, 17].

**Concerns and Risks**: Some concerns include high valuation, potential cooling in AI expenditures, and competition from Google's TPUs [5, 13, 7].

**Upcoming Events**: Investors are closely watching the Q4 2025 earnings scheduled for February 25, 2026, and announcements from CES 2026 [17, 20].

### Conclusion

Our internal thesis on NVIDIA aligns well with the current market sentiment. Both highlight NVIDIA's leadership in AI and data center markets, robust financial performance, and strong growth prospects. While there are some concerns about valuation and competition, the overall outlook remains positive, supporting our "STRONG BUY" rating.

---

## 📊 Observability

View detailed traces in LangSmith:
- Each query shows the **tool call sequence**
- See which tools were checked first vs escalated
- Monitor **latency** differences between internal vs external calls
- Track **token usage** for cost optimization

**Dashboard:** https://smith.langchain.com
